# 2048 Deep Q-Network
Training and evaluation notebook for the DQN agent.  
Works on **Google Colab** and **Kaggle** out of the box — no extra installs needed.

## 1 · Setup
Clone the repo and add it to the Python path so `game.py` and `dqn.py` are importable.

In [ ]:
!git clone https://github.com/<your-username>/RL2048.git
import sys
sys.path.insert(0, 'RL2048')

In [ ]:
import numpy as np
from game import TwntyFrtyEight
from dqn import (
    DQNAgent,
    plot_training_curve,
    plot_tile_distribution,
    compare_agents,
    _run_one,
)

## 2 · Train
Adjust `n_games` to taste. 2 000 is a reasonable starting point; 5 000–10 000 gives noticeably better results on a GPU runtime.

In [ ]:
agent = DQNAgent(
    lr=1e-4,
    hidden_size=128,
    buffer_capacity=10_000,
)

train_stats = agent.train(
    n_games=2000,
    epsilon_start=0.3,
    epsilon_end=0.05,
    epsilon_decay=0.995,
    batch_size=128,
    print_every=100,
)

In [ ]:
# Save weights so you can reload without retraining
agent.save('dqn_weights.pth')

## 3 · Training curve
Three panels: moves per game, total reward per game, and max tile per game.  
An upward trend in moves = the agent is learning to stay alive longer.

In [ ]:
plot_training_curve(train_stats, window=100)

## 4 · Evaluate
Play greedily (ε = 0) for `n_games` games and collect tile/step stats.

In [ ]:
eval_stats = agent.evaluate(n_games=200)
plot_tile_distribution(eval_stats, label='DQN')

## 5 · Compare agents
Run the SARSA and beam search policies for comparison.  

> **Beam search is slow** — each game at `depth=20, k=10` takes a few seconds.  
> Use `depth=10, k=5` for a quicker run, or reduce `n_games`.

In [ ]:
from agent import Agent

game  = TwntyFrtyEight()
sarsa = Agent(game)

# Load trained SARSA weights if available; otherwise uses zero weights (random-ish)
try:
    loaded = np.load('RL2048/w_star.npy')
    if loaded.shape == sarsa.w.shape:
        sarsa.w = loaded
        print(f'Loaded SARSA weights  shape={loaded.shape}')
    else:
        print('Shape mismatch — using zero weights')
except FileNotFoundError:
    print('w_star.npy not found — using zero weights')

In [ ]:
N_SARSA = 100   # greedy SARSA games
N_BEAM  = 30    # beam search games (slow — keep small)
BEAM_DEPTH = 10
BEAM_K     = 5

print(f'Running {N_SARSA} SARSA games…')
sarsa_stats = [_run_one(game, sarsa.greedy_policy) for _ in range(N_SARSA)]

print(f'Running {N_BEAM} beam search games (depth={BEAM_DEPTH}, k={BEAM_K})…')
beam_stats = [
    _run_one(game, lambda s: sarsa.beam_search_policy(s, depth=BEAM_DEPTH, k=BEAM_K))
    for _ in range(N_BEAM)
]

print('Done.')

In [ ]:
compare_agents({
    'DQN':              eval_stats,
    'SARSA':            sarsa_stats,
    f'Beam (d={BEAM_DEPTH},k={BEAM_K})': beam_stats,
})

## 6 · Resume training (optional)
Load saved weights and continue training from where you left off.

In [ ]:
agent2 = DQNAgent()
agent2.load('dqn_weights.pth')

more_stats = agent2.train(n_games=1000, epsilon_start=0.1, print_every=100)
agent2.save('dqn_weights.pth')

# Plot combined training history
all_stats = train_stats + [
    {**s, 'game': s['game'] + len(train_stats)} for s in more_stats
]
plot_training_curve(all_stats, window=100)